# 00 · Check inputs

Resolves every path pattern in `config/inputs.yaml`, opens each member, and reports anything
that would cause problems downstream. Nothing here modifies data.

1. Expected members and existence
2. Fixed fields (areacella, areacello, sftlf)
3. Per-member inventory and flags
4. Availability matrix
5. Fixed fields against the data grids
6. File metadata: model version and parentage
7. Branch consistency: pairing and variable identity
8. Save and summarise

A *member* is one (model, scenario, variable, realisation); it may span several files.
Outputs go to `interim_outputs/00_inputs/`.

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from inputs import (load_config, is_configured, expected_files, check_existence,
                    open_parts, standardise, load_member, open_nc, data_var, time_dim,
                    get_latlon, spatial_shape, grids_match, fx_path, path_exists,
                    time_position, month_index, model_roles, variable_grid)

CONFIG = os.environ.get("TOE_CONFIG", "inputs.yaml")
OUT = Path(os.environ.get("TOE_OUT", "interim_outputs/00_inputs"))
OUT.mkdir(parents=True, exist_ok=True)

N_WORKERS = 8   # parallel member opens

cfg = load_config(CONFIG)
MODELS = cfg["run"]["models"]
SCENARIOS = cfg["run"]["scenarios"]
CONFIGURED = [m for m in MODELS if is_configured(cfg, m)]
print("config:", Path(CONFIG).resolve())
print("configured:", CONFIGURED)
print("not configured:", [m for m in MODELS if m not in CONFIGURED])

## 1. Expected members and existence

In [ ]:
files = check_existence(expected_files(cfg))

(files[files.model.isin(CONFIGURED)]
     .pivot_table(index=["model", "scenario", "role", "var"], columns="status",
                  values="member", aggfunc="count", fill_value=0))

In [ ]:
# Missing members, with the pattern that found nothing
missing = files[files.status == "missing"]
(missing.groupby(["model", "scenario", "var"])
        .agg(missing_members=("member", ", ".join), example_pattern=("pattern", "first")))

## 2. Fixed fields

Area sums are compared with the expected global values in `fx_checks`. For sftlf, units
are inferred from the values and the Niño 3.4 box (5°S–5°N, 170°W–120°W) must be all ocean.

In [ ]:
def nino34_box(lat, lon):
    la, lo = xr.broadcast(lat, lon)
    lo = lo % 360
    return (la >= -5) & (la <= 5) & (lo >= 190) & (lo <= 240)

def fx_needed(model):
    # areacella and sftlf always; areacello only if some variable is on the ocean grid
    grids = {variable_grid(cfg, model, v) for v in model_roles(cfg, model).values()}
    return {"areacella", "sftlf"} | ({"areacello"} if "ocean" in grids else set())

fx_rows, fx_fields = [], {}
for model in CONFIGURED:
    for name in ("areacella", "areacello", "sftlf"):
        row = dict(model=model, field=name, path=fx_path(cfg, model, name),
                   provenance=(cfg["models"][model].get("fx_provenance") or {}).get(name))
        if row["path"] is None:
            row["status"] = "not provided" if name in fx_needed(model) else "not needed"
        elif not path_exists(row["path"]):
            row["status"] = "missing"
        else:
            ds = open_nc(row["path"])
            da = ds[data_var(ds, [name])].load()
            lat, lon = get_latlon(ds)
            fx_fields[(model, name)] = dict(da=da, lat=lat, lon=lon, shape=da.shape)
            row.update(status="ok", shape=da.shape, units=da.attrs.get("units"),
                       vmin=float(da.min()), vmax=float(da.max()), has_latlon=lat is not None)
            chk = (cfg.get("fx_checks") or {}).get(name)
            if chk:
                row["sum"] = float(da.sum())
                row["sum_rel_err"] = row["sum"] / float(chk["expected_sum"]) - 1
                if abs(row["sum_rel_err"]) > chk["rtol"]:
                    row["status"] = "check sum"
            if name == "sftlf":
                pct = row["vmax"] > 1.5
                row["units_inferred"] = "%" if pct else "fraction"
                if lat is not None:
                    row["nino34_max_land"] = float(da.where(nino34_box(lat, lon)).max())
                    if row["nino34_max_land"] > 0:
                        row["status"] = "land in Nino3.4 box"
                    trop = abs(xr.broadcast(lat, lon)[0]) <= 20
                    row["trop_cells_strict"] = int(((da == 0) & trop).sum())
                    row["trop_cells_majority"] = int(((da < (50 if pct else 0.5)) & trop).sum())
            ds.close()
        fx_rows.append(row)

fx_report = pd.DataFrame(fx_rows)
fx_report

## 3. Per-member inventory

Each present member is opened and standardised exactly as downstream notebooks will see it
(`inputs.standardise`: time fix, de-duplication, renaming, unit scaling). Recorded per member:

- files, total size, variable name inside the files
- **time stamping**: where the first stamp sits in its averaging interval (0 start,
  0.5 centre, 1 end), before any fix
- **overlaps**: time steps present in more than one file, and whether the copies agree
- **gaps**: breaks in the monthly sequence after de-duplication
- time span, calendar, units, value range on the first and last step, grid

In [ ]:
META_KEYS = ["source_id", "experiment_id", "variant_label", "parent_source_id",
             "parent_experiment_id", "parent_variant_label", "branch_time_in_parent"]

def infer_grid(model, shape):
    hits = [g for g, n in (("atmos", "areacella"), ("ocean", "areacello"))
            if (model, n) in fx_fields and fx_fields[(model, n)]["shape"] == tuple(shape)]
    return "|".join(hits) if hits else "unknown"

def overlaps(parts, row):
    """(n duplicated steps, copies agree?) across a member's files."""
    times = []
    for ds in parts:
        times.append(set(ds[time_dim(ds)].values))
    n_dup, agree = 0, True
    for i in range(len(parts)):
        for j in range(i + 1, len(parts)):
            common = sorted(times[i] & times[j])
            n_dup += len(common)
            if common:
                vi = data_var(parts[i], [row["file_var"], row["var"]])
                vj = data_var(parts[j], [row["file_var"], row["var"]])
                ti, tj = time_dim(parts[i]), time_dim(parts[j])
                a = parts[i][vi].sel({ti: common[0]}).values
                b = parts[j][vj].sel({tj: common[0]}).values
                agree &= bool(np.allclose(a, b, equal_nan=True))
    return n_dup, agree

def inspect(row):
    out = dict(key=row.name)
    try:
        parts = open_parts(row)
        out["meta"] = {k: parts[0].attrs.get(k) for k in META_KEYS}
        out["time_position"] = time_position(parts[0])
        out["n_dup"], out["dups_agree"] = overlaps(parts, row) if len(parts) > 1 else (0, True)
        out["nc_var"] = data_var(parts[0], [row["file_var"], row["var"]])
        ds = standardise(parts, row)
        da = ds[row["var"]]
        t = ds.time
        ym = month_index(t)
        ends = np.concatenate([np.ravel(da.isel(time=0).values), np.ravel(da.isel(time=-1).values)])
        out.update(n_time=len(t), t_start=str(t.values[0])[:10], t_end=str(t.values[-1])[:10],
                   calendar=getattr(t.values[0], "calendar", None),
                   n_gaps=int((np.diff(ym) != 1).sum()),
                   units=da.attrs.get("units"),
                   vmin=float(np.nanmin(ends)), vmax=float(np.nanmax(ends)),
                   spatial_shape=spatial_shape(da, "time"))
        for p in parts:
            p.close()
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out

present = files[files.status == "present"]
with ThreadPoolExecutor(N_WORKERS) as ex:
    details = pd.DataFrame(list(ex.map(inspect, [r for _, r in present.iterrows()]))).set_index("key")
inv = present.join(details)
if "error" not in inv:
    inv["error"] = None
inv["grid"] = [infer_grid(m, s) if isinstance(s, tuple) else None
               for m, s in zip(inv.model, inv.spatial_shape)]
print(f"inspected {len(inv)} members ({int(inv.n_files.sum())} files), "
      f"{inv.error.notna().sum()} errors")

In [ ]:
inv["issues"] = ""
inv["notes"] = ""
def flag(mask, text, col="issues"):
    inv.loc[mask, col] += text + "; "

flag(inv.error.notna(), "open error")
ok = inv.error.isna()

# Time stamping
pos = inv.time_position
flag(ok & (pos > 0.9) & (inv.time_fix != "center_of_bounds"),
     "stamped at END of averaging period but no time_fix")
flag(ok & pos.isna() & (inv.time_fix == "center_of_bounds"), "time_fix set but no time bounds")
flag(ok & pos.isna(), "no time bounds", col="notes")

# Overlaps and gaps
flag(ok & (inv.n_dup > 0) & ~inv.dups_agree.astype(bool), "overlapping files DISAGREE")
flag(ok & (inv.n_dup > 0) & inv.dups_agree.astype(bool),
     "duplicated steps removed (copies identical)", col="notes")
flag(ok & (inv.n_gaps > 0), "gaps in monthly sequence")

# Length, size and start relative to siblings (same model, scenario, variable)
key = ["model", "scenario", "var"]
mode_n = inv[ok].groupby(key).n_time.transform(lambda s: s.mode().iloc[0])
flag(ok & (inv.n_time != mode_n.reindex(inv.index)), "n_time differs from siblings")
med_size = inv.groupby(key).size_bytes.transform("median")
flag(inv.size_bytes < 0.9 * med_size, "total size >10% below siblings")
start_mode = inv[ok].groupby(["model", "scenario"]).t_start.transform(lambda s: s.mode().iloc[0])
flag(ok & (inv.t_start != start_mode.reindex(inv.index)), "start date differs within scenario")

# Units and ranges
for var, chk in cfg["variable_checks"].items():
    m = ok & (inv["var"] == var)
    flag(m & ~inv.units.isin(chk["units"]), f"units not in {chk['units']}")
    lo, hi = chk["range"]
    if var.startswith("siconc"):
        frac_units = inv.units.isin(["1", "fraction"])
        hi = np.where(frac_units, 1, hi)
        flag(m & (inv.vmax <= 1.5) & (inv.units == "%"), "values look like a fraction but units say %")
        flag(m & (inv.vmax > 1.5) & frac_units, "values look like % but units say fraction")
    flag(m & ((inv.vmin < lo) | (inv.vmax > hi)), "values out of range")

# Grid
flag(ok & (inv.grid != inv.grid_expected), "grid differs from expected")

for c in ("issues", "notes"):
    inv[c] = inv[c].str.rstrip("; ")
cols = ["model", "scenario", "var", "member", "n_files", "nc_var", "n_time", "t_start",
        "t_end", "time_position", "units", "grid", "issues"]
inv.loc[inv.issues != "", cols]

In [ ]:
# Overview: one row per model/scenario/variable
def uniq(s):
    return ",".join(sorted(set(map(str, s))))
(inv[ok].groupby(["model", "scenario", "var"])
        .agg(nc_var=("nc_var", uniq), units=("units", uniq), calendar=("calendar", uniq),
             grid=("grid", uniq), time_position=("time_position", uniq),
             t_start=("t_start", "min"), t_end=("t_end", "max"),
             n_members=("member", "count"), files_per_member=("n_files", uniq),
             notes=("notes", uniq)))

## 4. Availability matrix

One panel per model; rows are scenario × role, columns are members. "Present, flagged"
means the member exists but at least one check in section 3 failed.

In [ ]:
STATES = ["present", "present, flagged", "missing", "not available", "not required"]
COLOURS = ["#2e8b57", "#e5a50a", "#c0392b", "#bdbdbd", "#f2f2f2"]

state = files.copy()
state["state"] = state.status.str.replace("_", " ")
state.loc[inv.index[inv.issues != ""], "state"] = "present, flagged"

def panel_rows(model):
    s = state[state.model == model]
    roles = list(dict.fromkeys(s.role))
    keep = [sc for sc in SCENARIOS if not (s[s.scenario == sc].status == "not_required").all()]
    return s, roles, [(sc, r) for sc in keep for r in roles]

heights = [len(panel_rows(m)[2]) * 0.28 + 0.6 if m in CONFIGURED else 0.35 for m in MODELS]
fig, axes = plt.subplots(len(MODELS), 1, figsize=(9, sum(heights) + 0.6), squeeze=False,
                         gridspec_kw=dict(height_ratios=heights))
for ax, model in zip(axes[:, 0], MODELS):
    if model not in CONFIGURED:
        ax.text(0, 0.5, f"{model}: not yet configured", ha="left", va="center",
                fontsize=10, color="0.4", transform=ax.transAxes)
        ax.set_axis_off()
        continue
    s, roles, rows = panel_rows(model)
    members = cfg["models"][model]["members"]
    lookup = {(a, b, c): st for a, b, c, st in zip(s.scenario, s.role, s.member, s.state)}
    grid = np.array([[STATES.index(lookup.get((sc, r, m), "not available"))
                      for m in members] for sc, r in rows])
    ax.imshow(grid, cmap=ListedColormap(COLOURS), vmin=-0.5, vmax=len(STATES) - 0.5,
              aspect="auto")
    var_of = dict(zip(s.role, s["var"]))
    ax.set_yticks(range(len(rows)), [f"{sc} | {r} ({var_of[r]})" for sc, r in rows], fontsize=8)
    ax.set_xticks(range(len(members)), members, fontsize=8)
    ax.set_xticks(np.arange(-0.5, len(members)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(rows)), minor=True)
    ax.grid(which="minor", color="white", lw=1.5)
    ax.tick_params(which="minor", length=0)
    for k in range(len(roles), len(rows), len(roles)):
        ax.axhline(k - 0.5, color="black", lw=1)
    ax.set_title(model, loc="left", fontsize=11, fontweight="bold")

fig.legend([Patch(color=c) for c in COLOURS], STATES, loc="upper center",
           ncol=len(STATES), fontsize=8, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(OUT / "availability_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Fixed fields against the data grids

Shape matching (section 3) can pass when two grids differ only in latitude values
(e.g. Gaussian vs regular). This compares the actual coordinates of each fx file with one
member on the same grid.

In [ ]:
rows = []
for model in CONFIGURED:
    for grid_kind, name in (("atmos", "areacella"), ("atmos", "sftlf"), ("ocean", "areacello")):
        fx = fx_fields.get((model, name))
        ref = inv[(inv.model == model) & (inv.grid == grid_kind) & ok]
        row = dict(model=model, field=name, grid=grid_kind)
        if fx is None or ref.empty:
            row["result"] = "nothing to compare"
        else:
            ref = ref.iloc[0]
            ds = open_parts(ref)[0]
            match = grids_match(fx["lat"], fx["lon"], *get_latlon(ds))
            ds.close()
            row.update(reference=f"{ref.scenario}/{ref['var']}/{ref.member}",
                       result={True: "coordinates match", False: "COORDINATES DIFFER",
                               None: "shape match only (no coordinates)"}[match])
        rows.append(row)
grid_report = pd.DataFrame(rows)
grid_report

## 6. File metadata: model version and parentage

Global attributes from the first file of each member, where present (CMORised files carry
them; raw model output often does not). The main thing to check is that `source_id` is the
same model version for a SAI scenario and its baseline, and that `parent_variant_label`
points at the paired baseline member.

In [ ]:
meta = pd.DataFrame([{**{"model": r.model, "scenario": r.scenario, "var": r["var"],
                         "member": r.member}, **(r.meta or {})}
                     for _, r in inv[ok].iterrows()])
meta_cols = [c for c in META_KEYS if c in meta and meta[c].notna().any()]
meta_summary = (meta.groupby(["model", "scenario"])[meta_cols]
                    .agg(lambda s: ",".join(sorted(set(map(str, s.dropna())))) or "—")
                if meta_cols else pd.DataFrame())
meta_summary

## 7. Branch consistency

For each SAI scenario, the global-mean `gmsat` series from each SAI member is compared with
every baseline member from the month the SAI run starts.

- **Pairing.** If SAI rN branches from baseline rN, its first month should be much closer to
  that member than to any other. `pairing_ok` is True when the paired member gives the
  smallest RMS difference over the first `N_PAIR` months; `ratio_paired_unpaired` shows how
  clear-cut that is (a genuine pair should be well below 1). Longer windows are unreliable
  because members decorrelate within a few months.
- **Variable identity.** The mean SAI − paired-baseline difference over the first `N_ID`
  months should be near zero before injection has much effect. A systematic offset
  suggests the two series hold different quantities (e.g. skin vs near-surface air
  temperature). A ramp that starts in the first year will also produce a cooling offset.

In [ ]:
N_PAIR, N_ID, OFFSET_TOL = 1, 12, 0.3   # months, months, K

def global_mean_series(row, n_months, start_ym=None):
    ds = load_member(row)
    da = ds[row["var"]]
    da = da.assign_coords(ym=("time", month_index(da.time))).swap_dims(time="ym")
    if start_ym is not None:
        da = da.sel(ym=slice(start_ym, None))
    da = da.isel(ym=slice(0, n_months)).load()
    sdims = [d for d in da.dims if d != "ym"]
    area = fx_fields.get((row.model, "areacella"))
    if area is not None and area["shape"] == spatial_shape(da, "ym"):
        w = xr.DataArray(area["da"].values, dims=sdims)
    else:
        lat = get_latlon(ds)[0]
        w = np.cos(np.deg2rad(lat)) if lat is not None else None
    gm = da.weighted(w).mean(sdims) if w is not None else da.mean(sdims)
    return gm.reset_coords(drop=True)

branch_rows = []
for model in CONFIGURED:
    for scen in SCENARIOS:
        sc = cfg["scenarios"][scen] or {}
        if sc.get("kind") != "sai":
            continue
        g = inv[(inv.model == model) & (inv.role == "gmsat") & ok]
        sai, base = g[g.scenario == scen], g[g.scenario == sc["baseline"]]
        if sai.empty or base.empty:
            continue
        n = max(N_PAIR, N_ID)
        sai_gm = {r.member: global_mean_series(r, n) for _, r in sai.iterrows()}
        start = int(min(int(s.ym[0]) for s in sai_gm.values()))
        base_gm = {r.member: global_mean_series(r, n + 24, start_ym=start)
                   for _, r in base.iterrows()}
        for m, s in sai_gm.items():
            rms = {}
            for bm, b in base_gm.items():
                d = (s - b).dropna("ym").isel(ym=slice(0, N_PAIR))
                rms[bm] = float(np.sqrt((d ** 2).mean())) if d.size else np.nan
            best = min(rms, key=lambda k: rms[k] if np.isfinite(rms[k]) else np.inf)
            unpaired = [v for k, v in rms.items() if k != m]
            row = dict(model=model, scenario=scen, member=m,
                       sai_start=f"{start // 12}-{start % 12 + 1:02d}",
                       closest_baseline=best, rms_paired=rms.get(m, np.nan),
                       rms_median_unpaired=np.nanmedian(unpaired) if unpaired else np.nan)
            row["ratio_paired_unpaired"] = row["rms_paired"] / row["rms_median_unpaired"]
            row["pairing_ok"] = best == m
            if m in base_gm:
                d = (s - base_gm[m]).dropna("ym").isel(ym=slice(0, N_ID))
                row["mean_offset_first_N_ID"] = float(d.mean())
                row["offset_flag"] = abs(row["mean_offset_first_N_ID"]) > OFFSET_TOL
            branch_rows.append(row)

branch_report = pd.DataFrame(branch_rows)
branch_report

## 8. Save and summarise

In [ ]:
files.drop(columns=["files"]).to_csv(OUT / "expected_members.csv", index=False)
inv.drop(columns=["files", "meta"]).to_csv(OUT / "inventory.csv", index=False)
fx_report.to_csv(OUT / "fx_report.csv", index=False)
grid_report.to_csv(OUT / "grid_report.csv", index=False)
meta.to_csv(OUT / "file_metadata.csv", index=False)
branch_report.to_csv(OUT / "branch_report.csv", index=False)

print("Issues to resolve")
print("-----------------")
for m in [m for m in MODELS if m not in CONFIGURED]:
    print(f"- {m}: not configured")
for (model, scen, var), g in files[files.status == "missing"].groupby(["model", "scenario", "var"]):
    print(f"- {model} {scen} {var}: missing {', '.join(g.member)}")
for _, r in fx_report[~fx_report.status.isin(["ok", "not needed"])].iterrows():
    print(f"- {r.model} fx {r.field}: {r.status}")
for _, r in grid_report[grid_report.result.str.contains("DIFFER")].iterrows():
    print(f"- {r.model} {r.field}: coordinates differ from {r.reference}")
for (model, scen, var, iss), g in inv[inv.issues != ""].groupby(["model", "scenario", "var", "issues"]):
    print(f"- {model} {scen} {var} [{', '.join(g.member)}]: {iss}")
if "source_id" in meta:
    for model, g in meta[meta.source_id.notna()].groupby("model"):
        if g.source_id.nunique() > 1:
            print(f"- {model}: source_id differs across scenarios: "
                  + "; ".join(f"{s}={v}" for s, v in g.groupby("scenario").source_id.first().items()))
if len(branch_report):
    for _, r in branch_report[~branch_report.pairing_ok].iterrows():
        print(f"- {r.model} {r.scenario} {r.member}: closest baseline is {r.closest_baseline}, "
              f"not the paired member (ratio {r.ratio_paired_unpaired:.2f})")
    for _, r in branch_report[branch_report.get("offset_flag", False) == True].iterrows():
        print(f"- {r.model} {r.scenario} {r.member}: mean offset "
              f"{r.mean_offset_first_N_ID:+.2f} K vs paired baseline")

print("\nNotes (handled, for information)")
print("--------------------------------")
for (model, scen, var, n), g in inv[inv.notes != ""].groupby(["model", "scenario", "var", "notes"]):
    print(f"- {model} {scen} {var} [{', '.join(g.member)}]: {n}")